In [ ]:
import pandas as pd
import geopandas as gpd
import glob
from shapely.geometry import Point

# Path to the folder containing the CSV files
folder_path = './bikeshare_data/portland_bikeshare'
# Use glob to find all CSV files in the folder
csv_files = glob.glob(f"{folder_path}/*.csv")

combined_df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)
combined_df.columns

In [ ]:
# Combining the dataframe's two versions of lat/long columns, one named StartLatitude and one named Start_Latitude
combined_df["StartLatitude"] = combined_df["StartLatitude"].fillna(combined_df["Start_Latitude"])
combined_df["StartLongitude"] = combined_df["StartLongitude"].fillna(combined_df["Start_Longitude"])

# Combine the end columns
combined_df["EndLatitude"] = combined_df["EndLatitude"].fillna(combined_df["End_Latitude"])
combined_df["EndLongitude"] = combined_df["EndLongitude"].fillna(combined_df["End_Longitude"])

# Drop the extra columns
combined_df.drop(
    columns=[
        "Start_Latitude", 
        "Start_Longitude",
        "End_Latitude", 
        "End_Longitude", 
    ], 
    inplace=True
)

In [ ]:
# Read the shapefile of Oregon census tracts
tracts = gpd.read_file("./tract_shapefiles/tl_2023_or_tract/tl_2023_41_tract.shp")

# Create a GeoDataFrame for start locations
gdf_starts = gpd.GeoDataFrame(
    combined_df.copy(),
    geometry=gpd.points_from_xy(
        combined_df["StartLongitude"], 
        combined_df["StartLatitude"]
    ),
    crs="EPSG:4326"
)
gdf_starts = gdf_starts.to_crs(tracts.crs)

# Create a GeoDataFrame for end locations
gdf_ends = gpd.GeoDataFrame(
    combined_df.copy(),
    geometry=gpd.points_from_xy(
        combined_df["EndLongitude"], 
        combined_df["EndLatitude"]
    ),
    crs="EPSG:4326"
)
gdf_ends = gdf_ends.to_crs(tracts.crs)


import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,8))
tracts.plot(ax=ax, color='white', edgecolor='black')
gdf_starts.plot(ax=ax, marker='o', color='red', markersize=5)
# ax.set_xlim(-74.3, -73.7)
# ax.set_ylim(40.48, 40.95)
plt.show()

# Identify which tract each start and end is in
start_join = gpd.sjoin(gdf_starts, tracts, how="left", predicate="within")
end_join = gpd.sjoin(gdf_ends, tracts, how="left", predicate="within")

# Count how many trips landed in each tract (for starts and ends)
start_counts = start_join.groupby("GEOID").size().reset_index(name="start_count")
end_counts   = end_join.groupby("GEOID").size().reset_index(name="end_count")

# Combine both into one table
combined_counts = pd.merge(start_counts, end_counts, how="outer", on="GEOID").fillna(0)

# Single column of total trips (start + end)
combined_counts["total_trips"] = combined_counts["start_count"] + combined_counts["end_count"]

# Keep it tidy
aland_lookup = tracts[["GEOID", "ALAND"]].drop_duplicates()
combined_counts = combined_counts.merge(aland_lookup, on="GEOID", how="left")



# Keep only relevant columns and convert types
tracts_with_trips = combined_counts[["GEOID", "ALAND", "total_trips"]].astype({"total_trips": "int"})
# There are 731 days (inclusive) between March 1st, 2018 and February 29, 2020
tracts_with_trips['trips_per_day'] = tracts_with_trips['total_trips'] / 731
tracts_with_trips.to_csv('./bikeshare_data/tracts_with_trips/portland_tracts_with_trips.csv')


In [ ]:
census_files = ['./census_data/S0101_2023.csv',
                './census_data/S0601_2023.csv',
                './census_data/S0701_2023.csv',
                './census_data/S0801_2023.csv',
                './census_data/S0802_2023.csv',
                './census_data/S0601_2023.csv'
                ]

variable_meanings = {}

merged_oh_data = pd.read_csv(census_files[0])
merged_oh_data['GEOID'] = merged_oh_data['GEO_ID'].str.split('US').str[1]  # Extract GEOID from GEO_ID
merged_oh_data = merged_oh_data.drop(columns=['GEO_ID'])  # Drop redundant column

meanings = merged_oh_data.iloc[0].to_dict()  # Convert the first row to a dictionary
variable_meanings.update(meanings)  # Update the global dictionary of meanings
merged_oh_data = merged_oh_data.iloc[1:].reset_index(drop=True)

# Sequentially merge the remaining CSVs
for file in census_files[1:]:
    df = pd.read_csv(file)
    df['GEOID'] = df['GEO_ID'].str.split('US').str[1]  # Extract GEOID from GEO_ID
    df = df.drop(columns=['GEO_ID'])  # Drop redundant column
    df = df.drop(columns=['NAME'])
    meanings = df.iloc[0].to_dict()  # Convert the first row to a dictionary
    variable_meanings.update(meanings)  # Update the global dictionary of meanings
    df = df.iloc[1:].reset_index(drop=True)

    merged_oh_data = pd.merge(merged_oh_data, df, on='GEOID', how='outer')  # Merge on GEOID


# Merge with tracts_with_trips DataFrame
merged_df = pd.merge(tracts_with_trips, merged_oh_data, on='GEOID', how='left')

# Ensure all object columns are converted to numeric where possible
for col in merged_df.select_dtypes(include=['object']).columns:
    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')


# Exclude columns whose names end with M
columns_to_exclude = [col for col in merged_df.columns if col.endswith('M')]
filtered_columns = merged_df.columns.difference(columns_to_exclude)

# Select numeric columns that are not excluded
numeric_columns = merged_df[filtered_columns].select_dtypes(include=['number']).columns
merged_df = merged_df[numeric_columns]
merged_df['trips_per_day'] = merged_df['total_trips'] / 366

# List of columns to transform
columns_to_transform = [col for col in merged_df.columns if col not in ['GEOID', 'ALAND', 'total_trips', 'trips_per_day']]

# Creating new columns by dividing by population and area
for col in columns_to_transform:
    merged_df[f'{col}_per_capita'] = merged_df[col] / merged_df['S0101_C01_001E'] # total population
    merged_df[f'{col}_per_ALAND'] = merged_df[col] / merged_df['ALAND']

# Save combined dataframe for Multiple Regression analysis
merged_df.to_csv('./multiple_regression/or_census_data_and_trips.csv')

# Calculate correlation between bike trips and all other numeric columns
correlations = merged_df.corrwith(merged_df['total_trips'])
#correlations = merged_df[numeric_columns].corrwith(merged_df['total_trips'] / merged_df['ALAND'])

# Convert to a DataFrame for better readability and sort by absolute correlation
correlation_df = correlations.reset_index()
correlation_df.columns = ['Variable', 'Correlation']
correlation_df['Absolute_Correlation'] = correlation_df['Correlation'].abs()

# Sort by absolute correlation in descending order
sorted_correlation_df = correlation_df.sort_values(by='Absolute_Correlation', ascending=False)

# Extracting the relevant prefix for variable names
def extract_prefix(variable_name):
    for i, char in enumerate(variable_name):
        if char.upper() == 'E':
            return variable_name[:i + 1]
    return variable_name  # Return the whole variable name if 'E' is not found

# Apply the function to create a new column 'Meaning'
sorted_correlation_df['Meaning'] = sorted_correlation_df['Variable'].apply(lambda x: variable_meanings.get(extract_prefix(x), 'Unknown'))

# Display the top variables
columns_to_print = ['Meaning', 'Correlation']
print(sorted_correlation_df[columns_to_print])
